# Predicción de Cancelación de Servicios

**Problema:** ¿Se puede predecir si un servicio será cancelado antes de que ocurra?

**Tipo:** Clasificación binaria  
**Variable objetivo:** cancelado / no cancelado  
**Valor de negocio:** Identificar servicios en riesgo para acciones de retención proactiva.

> Dataset: servicios contratados con información de producto, jurisdicción,
> tipo de empresa y canal de adquisición. Sin datos personales.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing import clean_data
from src.features import build_features, fit_encoder, apply_encoder, fit_scaler, apply_scaler
from src.evaluation import plot_model_comparison, plot_confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df_raw = pd.read_csv('data/ot.csv')
df = clean_data(df_raw)
print(f'Dataset: {len(df):,} registros | Cancelados: {df["IS_CANCELED"].mean():.1%}')

## 1. ¿Cómo se distribuyen las cancelaciones por tipo de producto?

In [ ]:
cancel_by_cat = (
    df.groupby('product_category')['IS_CANCELED']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'cancel_rate', 'count': 'n'})
    .sort_values('cancel_rate')
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cancel_by_cat.index, cancel_by_cat['cancel_rate'], color='#4C72B0')
for bar, (_, row) in zip(bars, cancel_by_cat.iterrows()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"n={int(row['n']):,}", va='center', fontsize=9)
avg = df['IS_CANCELED'].mean()
ax.axvline(avg, color='red', linestyle='--', label=f'Promedio ({avg:.1%})')
ax.set_xlabel('Tasa de cancelación')
ax.set_title('Tasa de cancelación por categoría de producto')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Entrenamiento de modelos

In [ ]:
y = df['IS_CANCELED']
X = build_features(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

encoder = fit_encoder(X_train)
X_tr_enc = apply_encoder(X_train, encoder)
X_te_enc = apply_encoder(X_test,  encoder)
scaler   = fit_scaler(X_tr_enc)
X_tr_sc  = apply_scaler(X_tr_enc, scaler)
X_te_sc  = apply_scaler(X_te_enc, scaler)

MODELS = {
    'Perceptron':          Pipeline([('m', Perceptron(random_state=42, class_weight='balanced', max_iter=1000))]),
    'Logistic Regression': Pipeline([('m', LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced'))]),
    'Decision Tree':       Pipeline([('m', DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=8))]),
    'Random Forest':       Pipeline([('m', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1))]),
}

results = []
for name, clf in MODELS.items():
    clf.fit(X_tr_sc, y_train)
    yp   = clf.predict(X_te_sc)
    yprb = clf.predict_proba(X_te_sc)[:, 1] if hasattr(clf, 'predict_proba') else None
    auc  = roc_auc_score(y_test, yprb) if yprb is not None else None
    f1   = f1_score(y_test, yp, average='macro')
    results.append({'Model': name, 'clf': clf,
                    'Accuracy': accuracy_score(y_test, yp),
                    'F1_macro': f1, 'ROC_AUC': auc,
                    'F1_canceled': f1_score(y_test, yp, pos_label=1, average='binary')})
    auc_str = f'{auc:.3f}' if auc is not None else 'N/A'
    print(f'{name:22s}  F1={f1:.3f}  AUC={auc_str}')

## 3. Comparación de modelos

In [ ]:
plot_model_comparison(results)

## 4. ¿Qué variables predicen la cancelación?

In [ ]:
best = max([r for r in results if r['ROC_AUC']], key=lambda r: r['F1_macro'])
rf_model = best['clf']

importances = rf_model.named_steps['m'].feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title(f"Top 10 features — {best['Model']}")
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

## 5. Matriz de confusión — mejor modelo

In [ ]:
plot_confusion_matrix(best['Model'], best['clf'], X_te_sc, y_test)

## Conclusiones

- El **tipo de producto** es el predictor más importante: servicios digitales y suscripciones cancelan más que formación o agente registrado.
- La **antigüedad de la cuenta** al momento de la compra importa: cuentas nuevas cancelan más.
- El mejor modelo es **Random Forest** con ROC-AUC > 0.70 a pesar del desbalance de clases.
- La métrica principal es **F1-macro**, no accuracy, porque el dataset está desbalanceado.
- **Limitación principal:** sin señales de comportamiento post-compra (uso del servicio, logins). Agregarlas mejoraría el modelo significativamente.